In [4]:
# Projektarbeit: Simulation von geladenen Teilchen in einer Box

import numpy as np
import matplotlib.pyplot as plt
import csv
import params


# Projektarbeit: Simulation von geladenen Teilchen in einer Box

#1 Klassen definieren

class Partikel:
    
    def __init__(self, x, y, vx, vy, q, m):
        self.x = x          # Position in x-Richtung
        self.y = y          # Position in y-Richtung
        self.vx = vx        # Geschwindigkeit in x-Richtung
        self.vy = vy        # Geschwindigkeit in y-Richtung
        self.q = q          # Ladung
        self.m = m          # Masse
        self.s = np.array([x,y,vx,vy]) #state vektor



class Box:
    def __init__(self, breite = 100, hoehe = 100):
        self.breite = breite          # Breite der Box
        self.hoehe = hoehe            # Höhe der Box
        self.partikel = []            # Liste der Partikel in der Box

    def add_partikel(self, partikel):
        # Fügt ein Partikel zur Box hinzu
        self.partikel.append(partikel)

    def bewegungs_dgl(self, state, m, q, g, partikel_liste):
    # berechnet die Ableitungen des State-Vektors für ein Partikel. Aktueller State-Vektor (x, v, vx, vy)


        x, y, vx, vy = state
        fx, fy = 0.0, m * g # Gravitationskraft wirkt konstant nach unten

    # Berechnung der elektrischen Kräfte durch andere Partikel

        for andere in partikel_liste:  # durchläuft jedes Teilchen (andere) in der Liste partikel_liste. Die Liste enthält alle Teilchen, die sich in der Box befinden.
            if andere.x == x and andere.y == y: # Bedingung überspringt das Teilchen selbst, da ein Partikel keine Kraft auf sich selbst ausübt
                continue 
            dx = x - andere.x   # Berechnet die Differenz in den x- und
            dy = y - andere.y  # y-Koordinaten zwischen dem aktuellen Teilchen (mit Position x,y) und dem anderen Teilchen
            abstand = np.sqrt(dx**2 + dy**2)

            if abstand > 1e-6: # Division durch null vermeiden (10^-6 wird als ausreichend klein betrachtet)
                f_elektrisch = (q* andere.q / (abstand**3))
                fx += f_elektrisch * dx
                fy += f_elektrisch * dy

        return np.array([vx, vy, fx / m, fy / m]) # Rückgabe der Ableitungen


        # Runge-Kutta-Verfahren
        
    def rk4_step(self, dgl, state, dt, m, q, g):
    
        k1 = dt * dgl(state, m, q, g, self.partikel)
        k2 = dt * dgl(state + 0.5 * k1, m, q, g, self.partikel)  
        k3 = dt * dgl(state + 0.5 * k2, m, q, g, self.partikel)  
        k4 = dt * dgl(state + k3, m, q, g, self.partikel)  

    
        return state + (k1 + 2 * k2 + 2 * k3 + k4) / 6  


    def check_kollisionen(self):
        # Überprüft und behandelt Kollisionen mit den Wänden
        for partikel in self.partikel:
            # Kollision mit der linken oder rechten Wand
            if partikel.x <= 0 or partikel.x >= self.breite:
                partikel.vx *= -1  # Reflexion: Geschwindigkeit in x-Richtung umkehren
            if partikel.y <= 0 or partikel.y>= self.hoehe:
                partikel.vy *= -1 # Reflexion: geschwindigkei in y-Richtung umkehren

    def update(self, dt):
        # Führt einen Simulationsschritt aus
        for partikel in self.partikel:
            # aktuellen Statevektor für Partikel erstellen
            state = np.array([partikel.x, partikel.y, partikel.vx, partikel.vy])
            # neuen state vektor mit Runge-Kutta berechnen
            neuer_state = self.rk4_step(self.bewegungs_dgl, state, dt, partikel.m, partikel.q, -10.0)
            # Position und Geschwindigkeit des partikels aktualisieren
            partikel.x, partikel.y,partikel.vx, partikel.vy = neuer_state
            
            # auf Kollision überprüfen
        self.check_kollisionen()

# Simulation der Teilchen

# Definition der Anfangszustände der Teilchen (x, y, vx, vy)
initial_states = [
    [1.0, 45.0, 10.0, 0.0],  # Teilchen 1
    [99.0, 55.0, -10.0, 0.0],  # Teilchen 2
    [10.0, 50.0, 15.0, -15.0],  # Teilchen 3
    [20.0, 30.0, -15.0, -15.0],  # Teilchen 4
    [80.0, 70.0, 15.0, 15.0],  # Teilchen 5
    [80.0, 60.0, 15.0, 15.0],  # Teilchen 6
    [80.0, 50.0, 15.0, 15.0]   # Teilchen 7
]

# Berechnung der Gesamtenergie (kinetisch + potenziell)
def berechne_energie(state, m, g):
    x, y, vx, vy = state
    kinetische_energie = 0.5 * m * (vx**2 + vy**2)
    potenzielle_energie = m * g * y
    return kinetische_energie + potenzielle_energie

# Simulation der Teilchen
def simuliere_teilchen(initial_states, m, q, g, dt, t_max, dateiname="output.csv"):
    # Datei für Output öffnen
    with open(dateiname, "w", newline="") as file:
        writer = csv.writer(file)

        # Kopfzeile schreiben
        kopfzeile = ["Zeit", "Energie"]
        for i in range(len(initial_states)):
            kopfzeile.extend([f"x{i+1}", f"y{i+1}", f"vx{i+1}", f"vy{i+1}"])
        writer.writerow(kopfzeile)
        print("Kopfzeile geschrieben:", kopfzeile)

        # Simulation starten
        t = 0.0
        zustand = np.array(initial_states)  # Anfangszustände der Teilchen

        while t <= t_max:
            # Berechnung der Gesamtenergie des Systems
            gesamtenergie = sum(berechne_energie(state, m, g) for state in zustand)

            # Zeile für den aktuellen Zeitpunkt erstellen
            zeile = [t, gesamtenergie]
            for state in zustand:
                zeile.extend(state)
            writer.writerow(zeile)

            # Runge-Kutta-Schritte für alle Teilchen durchführen
            for i in range(len(zustand)):
                zustand[i] = rk4_step(dgl, zustand[i], dt, m, q, g)

            # Zeit aktualisieren
            t += dt

    print(f"Simulation abgeschlossen. Ergebnisse in {dateiname} gespeichert.")